In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import numpy as np
import json
import matplotlib.pyplot as plt
from scipy.stats import lognorm
from standes.intensitymeasures import AverageSpectralAcceleration
from phd_project.config import config

cfg = config.load_config()

In [3]:
N_STOREYS = [3]
ANALYSIS_ROOT = cfg["analysis_data"]["wp1_casestudy_sites"]
RESULTS_ROOT = cfg["results"]["site_fragility_curves"]
FRAG_ROOT = cfg["proc_data"]["wp1_sites_fragility_curves"]
STRIPE_IML_PATH = Path(r"C:\Users\clemettn\Documents\phd\data_processed\05_gcim_distributions\imls_for_selection_AvgSA_03.json")

G = 9810    # mm/s²

PC_UPPER_THRESHOLD = 0.7 # the minimum upper collapse probabilty to consider a fragility curve as well defined.
PC_LOWER_THRESHOLD = 0.2 # the maximum lower collapse probabilty to consider a fragility curve as well defined.

# im
IM = AverageSpectralAcceleration(0, 3, n_periods=10)

def structure_tag(site_idx: int, n: int) -> str:
    return f"{n}s_cbf_dc2_site{site_idx}"

In [4]:
# Load the stripe IMLs for the MSA
with open(STRIPE_IML_PATH, "r") as file:
    stripe_imls = json.load(file)

## Load Fragility Curves

In [5]:
sites = list(range(0, 60))

not_well_defined = {"lt_upper_threshold": [],
                    "gt_lower_threshold": []}

# fragility_curves = {}

for site in sites:#[34:35]:
    site_fcs = {}
    for ns in N_STOREYS:
        # building tag
        tag = structure_tag(site, ns)
        save_folder = RESULTS_ROOT / f"site_{site}" / f"{ns}s"
        save_folder.mkdir(parents=True, exist_ok=True)

        # load a fragility curve
        fc_path = ANALYSIS_ROOT / f"site_{site}" / f"{ns}s/mdof/msa_AvgSA_03/collapse_fragility.json"
        if not fc_path.parent.exists():
            print(f"No MSA Data for site {site} and {ns}s. Skipping...")
            continue

        try:    
            with open(fc_path, "r") as file:
                fc = json.load(file)
                # resave it in the processed data folder
                save_path = FRAG_ROOT / f"site_{site}" / f"{tag}_msa_collapsefragility_AvgSA_03.json"
                with open(save_path, "w") as file:
                    json.dump(fc, file, indent=4)

                fc = {k: np.array(v) if isinstance(v, list) else v for k, v in fc.items()}

        except FileNotFoundError:
            print(f"No Fragility Curve for MSA: site {site} and {ns}s. Skipping...")
            continue

        if max(fc["efc"][1, :]) <= PC_UPPER_THRESHOLD:
            not_well_defined["lt_upper_threshold"].append((site, ns))

        if min(fc["efc"][1, :]) >= PC_LOWER_THRESHOLD:
            not_well_defined["gt_lower_threshold"].append((site, ns))

        im_max = lognorm.ppf(0.99, s=fc["dispersion"], scale=fc["median"])
        imls = np.linspace(0, im_max*1.15, 50) # in g
        
        msa_fc_fit = lognorm.cdf(imls, s=fc["dispersion"], scale=fc["median"])

        # get the stripe imls
        building_stripe_imls = stripe_imls[str(site)][tag]

        # plot fragility curve
        fig, ax = plt.subplots(figsize=(6, 4))
        plt.close(fig)

        for siml in building_stripe_imls:
            if siml is not None:
                ax.axvline(siml, ls="--", color="k", alpha=0.5)

        ax.plot(imls, msa_fc_fit, color="b", label="MSA")
        ax.plot(fc["efc"][0, :], fc["efc"][1, :], ls="none", marker="o", mfc="b", mec="k", alpha=0.75, label="MSA ecdf")

        ax.set_ylim(0, 1)
        ax.grid(ls="-.", color="0.8")
        ax.set_xlabel("AvgSA[0,3], [g]")
        ax.set_ylabel("Probability of Collapse, P[C]")
        ax.set_title(f"Fragility Curves - Site {site}, {ns}s")
        leg = ax.legend()
        frame = leg.get_frame()
        frame.set_edgecolor("k")
        fig.savefig(RESULTS_ROOT / f"fragility_curves_site_{site}_{ns}s.jpg", dpi=300, bbox_inches="tight")


print(f"\nSites with not well defined fragility curves (P[C] < {PC_UPPER_THRESHOLD}):\n {not_well_defined['lt_upper_threshold']}\n")
print(f"Sites with not well defined fragility curves (P[C] > {PC_LOWER_THRESHOLD}):\n {not_well_defined['gt_lower_threshold']}")


No MSA Data for site 24 and 3s. Skipping...
No MSA Data for site 31 and 3s. Skipping...
No MSA Data for site 38 and 3s. Skipping...

Sites with not well defined fragility curves (P[C] < 0.7):
 [(1, 3), (2, 3), (3, 3), (4, 3), (10, 3), (13, 3), (15, 3), (16, 3), (27, 3), (30, 3), (32, 3), (33, 3), (34, 3), (35, 3), (36, 3), (42, 3), (43, 3), (48, 3), (50, 3), (53, 3)]

Sites with not well defined fragility curves (P[C] > 0.2):
 [(4, 3), (12, 3), (17, 3), (22, 3), (23, 3), (27, 3), (28, 3), (37, 3), (44, 3), (49, 3), (57, 3), (58, 3)]


In [6]:
len(set(not_well_defined['lt_upper_threshold'] + not_well_defined['gt_lower_threshold']))

30